# Comparing world versions

Every model version of the world, graded on one fixed answer key, side by side. `world.ipynb` looks at
one version in detail; this notebook ranks versions against each other.

**The answer key.** The first 16 test samples of the `world_master` dataset (generate it in
`notebooks/world/data.ipynb`). Each sample is a 512² region with its own terrain and erosion parameters,
simulated whole. Each version's world is built on the same terrain and parameters, queried on the same
region, and graded on the interior, 64 cells in from the simulator's fixed edges. No version trains on
these samples.

**Adding a version.** Train its two levels under new profile names, then add one line to `VERSIONS`. Each
version's predictions are saved in `artifacts/compare/`, so only new or retrained versions are computed.


In [ ]:
import numpy as np

from erosion_sim.world.compare import Version, compare_versions, example_index
from erosion_sim.world.evaluate import show_examples

# name: Version(coarse profile, fine profile, sampler settings). Checkpoints are
# artifacts/checkpoints/diffusion-<profile>.pt.
VERSIONS = {
    "v0: closed 128² runs": Version("world_coarse", "world_fine"),
    "v0, 16 steps bf16": Version("world_coarse", "world_fine", steps=16, bf16=True),
}
DTS = (10_000.0, 30_000.0, 50_000.0)
MARGIN = 64

## Scores

Every version against the simulator's whole-region run, with these baselines:

- **no change:** the starting terrain. Any model must beat it.
- **`<version>, coarse level only`:** that version's coarse change, smoothly upsampled, with no fine
  level. It shows what the fine level adds.

The table is height error and river quality. The plot breaks the scores down by elapsed time. Seams and
speed are measured for one version in `world.ipynb`.

In [ ]:
result = compare_versions(VERSIONS, dTs=DTS, margin=MARGIN, coarse_only=True)

## Maps

Three samples at 30,000 yr. Top row of each: the starting terrain, then `Δz` of the simulator and of each
method, on one color scale. Bottom row: each method's error (method minus simulator).


In [ ]:
runs = [s["run"] for s in result["runs"]]
show_examples(result["pairs"], list(VERSIONS), examples=[example_index(result, run, 30_000.0) for run in runs[:3]],
              title="Each version against the simulator's whole-region run, dT 30,000 yr",
              simulator="simulator, whole region")

## The samples

Each sample's parameters, and each method's RMSE (m) on it at 30,000 yr. A version that wins on average
but loses badly on some parameters shows up here.


In [ ]:
pairs = result["pairs"]
samples = result["runs"]
idx = [example_index(result, s["run"], 30_000.0) for s in samples]
w = max(len(m) for m in pairs["methods"])
for label, key in [("K_ref", lambda s: f"{s['K_ref']:.1e}"), ("D / K", lambda s: f"{s['D_over_K']:.2f}"),
                   ("m", lambda s: f"{s['m']:.2f}"), ("persistence", lambda s: f"{s['terrain']['persistence']:.2f}"),
                   ("noise", lambda s: s["terrain"]["noise"][:7])]:
    print(f"{label:>{w}}  " + " ".join(f"{key(s):>7}" for s in samples))
print(f"{'sample':>{w}}  " + " ".join(f"{s['run']:>7}" for s in samples))
for m in pairs["methods"]:
    print(f"{m:>{w}}  " + " ".join(f"{np.sqrt(((pairs[m][i] - pairs['z_b'][i]) ** 2).mean()):7.2f}" for i in idx))